# Experiment: people detector - yolo11n vs RF-DETR Nano vs RF-DETR Small

**Only the player detector changes.** Ball model, thresholds and pass logic are identical.
The code comes from the branch `exp/people-detector-rfdetr`; this notebook only runs it.

Before running, in Google Drive (My Drive):
- `footvolley/models/ball_v2.pt`
- `footvolley/clips/rally_56s.mp4`

Runtime: Runtime > Change runtime type > **T4 GPU**. Then Runtime > Run all.
At the end, everything is saved to `My Drive/footvolley/outputs/exp_people_<date>/` and a zip is downloaded.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/footvolley'
for p in ['models/ball_v2.pt', 'clips/rally_56s.mp4']:
    print(('OK      ' if os.path.exists(f'{DRIVE}/{p}') else 'MISSING ') + p)

In [ ]:
BRANCH = 'exp/people-detector-rfdetr'
!rm -rf /content/repo
!git clone -q -b {BRANCH} https://github.com/YuvalMatyovis/footvolley-mlops-pipeline.git /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
!pip install -q ultralytics==8.4.191 rfdetr
from importlib.metadata import version
for pkg in ['ultralytics', 'rfdetr', 'torch']:
    print(pkg, version(pkg))

In [ ]:
!mkdir -p footvolley/models footvolley/data/clips
!cp {DRIVE}/models/ball_v2.pt footvolley/models/
!cp {DRIVE}/clips/rally_56s.mp4 footvolley/data/clips/
!ls -la footvolley/models footvolley/data/clips

## Smoke test (30 frames per backend)
Catches setup problems in a minute instead of after a full run. Each line should show players found (avg people > 0).

In [ ]:
CLIP = 'footvolley/data/clips/rally_56s.mp4'
BACKENDS = ['yolo11n', 'rfdetr-nano', 'rfdetr-small']
for b in BACKENDS:
    !python footvolley/src/perception.py --clip {CLIP} --people-backend {b} --people-model yolo11n.pt --max-frames 30 --out-root /content/smoke 2>&1 | grep -E "backend|WARNING|Error|error"
!cat /content/smoke/*rfdetr-small*/meta.json | grep -A3 rfdetr_

## Full runs: perception -> count_passes -> evaluate_passes -> render_events, once per backend

In [ ]:
import glob, subprocess
TRUTH = 'footvolley/ground_truth/rally_56_manual_gavish_2026-11-10.json'
RUNS = {}
for b in BACKENDS:
    print(f'\n########## {b} ##########')
    subprocess.run(['python', 'footvolley/src/perception.py', '--clip', CLIP, '--people-backend', b,
                    '--people-model', 'yolo11n.pt', '--progress-every', '500'], check=True)
    run = sorted(glob.glob(f'footvolley/outputs/rally_56s_{b}_*'))[-1]
    RUNS[b] = run
    subprocess.run(['python', 'footvolley/src/count_passes.py', '--run', run], check=True)
    subprocess.run(['python', 'footvolley/src/evaluate_passes.py', '--run', run, '--truth', TRUTH], check=True)
    subprocess.run(['python', 'footvolley/src/render_events.py', '--video', CLIP, '--run', run], check=True)
print(RUNS)

In [ ]:
!python footvolley/src/compare_people.py {' '.join(RUNS[b] for b in BACKENDS)} --out footvolley/outputs/comparison_people.md

## Save to Drive and download
Then drag the zip into Codespaces (footvolley/outputs/) and unzip there for the analysis.

In [ ]:
from datetime import datetime
from google.colab import files
name = 'exp_people_' + datetime.now().strftime('%Y-%m-%d_%H%M')
!mkdir -p {DRIVE}/outputs/{name}
!cp -r footvolley/outputs/* {DRIVE}/outputs/{name}/
!cd footvolley/outputs && zip -qr /content/{name}.zip .
!ls -la /content/{name}.zip
files.download(f'/content/{name}.zip')